# VoiceStudio · Kaggle 免费 GPU · Qwen3-TTS 语音克隆
使用 Kaggle 免费 Notebook GPU **离线生成一个 WAV 文件**，再从手机下载导入 VoiceStudio。它不是持久在线 API。

**手机操作：** Kaggle → New Notebook → Settings → Accelerator → GPU、Internet → On；上传自己录制并获授权的 3—10 秒 WAV 到 Notebook Dataset，填入下面的路径。每运行一次可能消耗 Kaggle GPU 配额。


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "soundfile"]) 
import torch, numpy as np, soundfile as sf
from pathlib import Path
assert torch.cuda.is_available(), "请先在 Kaggle Notebook 设置中启用 GPU"
print("Cloud GPU:", torch.cuda.get_device_name(0))


## 配置参考音频与合成文本
`REFERENCE_WAV` 填你通过 Kaggle 数据集上传后的绝对路径，一般是 `/kaggle/input/数据集名/reference.wav`。**不要上传未经授权的他人声音。**


In [ ]:
REFERENCE_WAV = "/kaggle/input/my-voice/reference.wav"  # 修改为自己上传的 WAV 路径
REFERENCE_TRANSCRIPT = "这是参考录音中的准确原文"         # 精确录音文字；可留空但克隆模式不同
TARGET_TEXT = "你好，这是 VoiceStudio 在 Kaggle 免费 GPU 上生成的声音。"
LANGUAGE = "Chinese"  # 可用 English, Japanese 等模型支持的语言
OUTPUT_WAV = "/kaggle/working/VoiceStudio-Qwen3-TTS.wav"

path = Path(REFERENCE_WAV)
assert path.is_file(), f"没有找到参考 WAV，请检查 Kaggle Input 路径: {path}"
assert 0 < len(TARGET_TEXT) <= 180, "先测试 1—180 字的短句"
details = sf.info(str(path))
assert 3 <= details.duration <= 10.01, "参考录音必须为 3—10 秒"
print("reference duration (s):", round(details.duration, 2))


In [ ]:
from qwen_tts import Qwen3TTSModel
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0",
    dtype=dtype,
)
waveforms, rate = model.generate_voice_clone(
    text=TARGET_TEXT,
    language=LANGUAGE,
    ref_audio=str(path),
    ref_text=REFERENCE_TRANSCRIPT.strip() or None,
    x_vector_only_mode=not bool(REFERENCE_TRANSCRIPT.strip()),
)
assert len(waveforms) == 1, "模型未返回完整合成音频"
audio = waveforms[0]
if hasattr(audio, "detach"):
    audio = audio.detach().float().cpu().numpy()
audio = np.asarray(audio, dtype=np.float32).reshape(-1)
assert audio.size > 0 and np.isfinite(audio).all(), "模型返回无效音频"
sf.write(OUTPUT_WAV, np.clip(audio, -1.0, 1.0), rate, subtype="PCM_16")
print("完成：", OUTPUT_WAV)
print("时长 (秒)：", round(len(audio) / rate, 2))


## 下载并导入 VoiceStudio
在 Kaggle 右侧 **Output / Files** 下载 `VoiceStudio-Qwen3-TTS.wav`。打开 VoiceStudio → 作品 → 导入 WAV。模型在 Kaggle 云端 GPU 执行，手机不安装模型。

**注意：** Kaggle 免费 Notebook 会休眠、断开或额度耗尽，因此不能作为 VoiceStudio 的持久在线 API；不要用暴露的隧道绕过它的资源限制。
